In [ ]:
# R1. Repaired-feedback pilot: fixed source, fixed seeds, unchanged previous experiment.
from pathlib import Path
import json,time,copy,hashlib,math,re,io,base64,random,html,traceback
from collections import Counter
import numpy as np,pandas as pd,torch
import torch.nn.functional as F
from PIL import Image,ImageDraw
from IPython.display import display,Markdown,HTML
assert all(k in globals() for k in ['D_PIPE','D_CLEAN','W_VM','W_VP','w_eval','W_NEUTRAL','_d_key']), 'Share the existing Hands_Jev_World_Knowledge_Feedback kernel; do not rerun the old experiment.'
R_OLD_OUT=Path('results/jev_world_feedback_20260922_102451')
R_OUT=Path('results')/time.strftime('jev_feedback_repairs_%Y%m%d_%H%M%S');R_OUT.mkdir(parents=True)
R_SEEDS=[123200100,123200101]
R_CONFIG={'source':str(D_SOURCE_PATHS[0]),'source_sha256':hashlib.sha256(D_SOURCE.tobytes()).hexdigest(),'seeds':R_SEEDS,'schedule_indices':[50,99],'rounds_per_seed':10,'advance_per_round':5,'probe_horizon':6,'Jev_stages':['hypothesis','hypothesis_conditioned_joint_parameters','matched_branch_selection'],'planned_controller_calls':60,'max_requests_including_tests_and_repeated_probe_responses':100,'edits_forced':False,'review_views':'paired whole images plus identically located model-derived context crops','history':'all current-seed branch summaries including rejected; detailed recent evidence plus exact ledger pointers','starting_pixels_replaced':False,'comparison_scope':'two-seed engineering pilot, not a quality benchmark or isolated ablation'}
R_OLD_MANIFEST_HASH=hashlib.sha256((R_OLD_OUT/'protocol.json').read_bytes()).hexdigest()
R_COUNTS={'jev_calls':0,'jev_seconds':0.,'vision_calls':0,'vision_seconds':0.};R_LEDGER=[];R_ROUNDS=[];R_FINALS={};R_FAILURES=[];R_RUNNING=False
R_PRE_FORWARD_COUNTS=copy.deepcopy(W_COUNTS)
def r_dump(name,obj):
 text=json.dumps(obj,indent=2,default=lambda x:x.tolist() if hasattr(x,'tolist') else str(x))
 assert _d_key not in text
 (R_OUT/name).write_text(text)
r_dump('protocol.json',R_CONFIG);D_SOURCE.save(R_OUT/'source.png')
display(Markdown('''# Jev feedback repairs: two-seed pilot
The same original seed-123 source and the first two prior re-noising seeds. This test repairs evidence memory, makes hypotheses precede control selection, adds paired/context views and reviewer sanity tests, and observes candidate trajectories for up to six steps. The outer loop still completes 50 steps per seed. No edit is accepted merely to make the test positive. All rejected branches, errors and decisions are archived.'''))
print('Output:',R_OUT,'GPU free GiB:',round(torch.cuda.mem_get_info()[0]/2**30,2))


In [ ]:
# R2. Paired pixel views and exact reviewer prompts, with independently checkable output.
from transformers import AutoProcessor
R_VP=AutoProcessor.from_pretrained(W_VLM_PATH,min_pixels=128*28*28,max_pixels=768*28*28,local_files_only=True)
R_PAIR_PROMPT='''The board shows two candidate images A and B: whole images in the top row, enlarged context from the SAME coordinates in the bottom row. Compare corresponding pixels and visible relationships, not the labels. They may be identical. First decide whether you can see a difference. Inspect finger-to-palm continuity, relative widths and lengths allowing perspective, joint bends, contour order at overlaps, thumb/palm relationships, and material or lighting continuity. Do not demand five visible fingers. Identify a preference only when a concrete relationship supports it; generic sharpness or skin detail is not enough. Report any collateral change and what cannot be assessed. Return only a compact JSON object with these keys: "difference" ("none", "visible", or "uncertain"), "preference" ("A", "B", "tie", or "uncertain"), "observations" (up to three short concrete comparisons with approximate locations), "collateral" (short text), "uncertainty" (short text). Do not infer which method made either image.'''
R_SINGLE_PROMPT='''The left panel is the whole current image and the right panel is an enlarged contextual crop of the SAME image. Explain its visible spatial relationships, not just its category. Describe finger/palm attachments, thumb/palm relation, plausible overlap or foreshortening, and any specific ambiguous or inconsistent boundary. Do not infer hidden fingers or invent a defect. Return only compact JSON with "relationships" (up to three short observations with approximate positions), "suspected_issue" (one short description or "none supported"), "alternative_explanation" (a plausible perspective/occlusion account, or "unresolved"), and "uncertainty". No proposed model controls.'''
R_VISION_CACHE={}
def r_context_box(mask,size=(512,512)):
 m=mask.detach().float().cpu().squeeze().numpy();m=np.maximum(m,0)
 # The same model-derived context window is used for both branches; no hand-drawn finger box.
 yy,xx=np.indices(m.shape);mass=max(m.sum(),1e-8)
 cx=float((m*xx).sum()/mass)/(m.shape[1]-1)*size[0];cy=float((m*yy).sum()/mass)/(m.shape[0]-1)*size[1]
 side=320;x=int(np.clip(round(cx-side/2),0,size[0]-side));y=int(np.clip(round(cy-side/2),0,size[1]-side))
 return (x,y,x+side,y+side)
def r_pair_board(a,b,box):
 board=Image.new('RGB',(768,816),(32,32,32));draw=ImageDraw.Draw(board)
 for j,(im,label) in enumerate([(a,'A'),(b,'B')]):
  draw.text((j*384+10,6),label+' whole image',fill='white');board.paste(im.resize((384,384)),(j*384,24))
  draw.text((j*384+10,410),label+' enlarged context',fill='white');board.paste(im.crop(box).resize((384,384)),(j*384,432))
 return board
def r_single_board(im,box):
 board=Image.new('RGB',(768,408),(32,32,32));d=ImageDraw.Draw(board)
 d.text((10,6),'Whole image',fill='white');d.text((394,6),'Enlarged context',fill='white')
 board.paste(im.resize((384,384)),(0,24));board.paste(im.crop(box).resize((384,384)),(384,24));return board
@torch.inference_mode()
def r_vision(board,prompt,tag):
 key=hashlib.sha256(board.tobytes()+prompt.encode()).hexdigest()
 if key in R_VISION_CACHE:return copy.deepcopy(R_VISION_CACHE[key])
 board.save(R_OUT/f'{tag}_views.png');t=time.time();R_COUNTS['vision_calls']+=1
 template=R_VP.apply_chat_template([{'role':'user','content':[{'type':'image'},{'type':'text','text':prompt}]}],tokenize=False,add_generation_prompt=True)
 inputs=R_VP(text=[template],images=[board],padding=True,return_tensors='pt').to('cuda')
 out=W_VM.generate(**inputs,max_new_tokens=300,do_sample=False,temperature=None,top_p=None,top_k=None)
 raw=R_VP.batch_decode(out[:,inputs.input_ids.shape[1]:],skip_special_tokens=True,clean_up_tokenization_spaces=False)[0]
 try:
  lo=raw.index('{');hi=raw.rindex('}')+1;parsed=json.loads(raw[lo:hi]);valid=isinstance(parsed,dict)
 except (ValueError,json.JSONDecodeError):parsed=None;valid=False
 result={'raw':raw,'parsed':parsed,'valid_JSON':valid,'view_file':f'{tag}_views.png','seconds':time.time()-t,'source':'Qwen2-VL-2B, observational report not anatomical ground truth'}
 R_COUNTS['vision_seconds']+=result['seconds'];R_VISION_CACHE[key]=copy.deepcopy(result);r_dump(f'{tag}_vision.json',result)
 return result
def r_pair(a,b,box,tag):
 result=r_vision(r_pair_board(a,b,box),R_PAIR_PROMPT,tag)
 result['exact_pixel_identity']=bool(np.array_equal(np.asarray(a),np.asarray(b)))
 result['crop_xyxy']=list(box)
 result['pixel_MAE']=float(np.abs(np.asarray(a).astype('float32')-np.asarray(b).astype('float32')).mean()/255)
 return result
r_dump('reviewer_prompts.json',{'paired':R_PAIR_PROMPT,'single':R_SINGLE_PROMPT})
display(Markdown('### Paired reviewer prompt\n'+R_PAIR_PROMPT))
display(Markdown('### Current-state reviewer prompt\n'+R_SINGLE_PROMPT))


In [ ]:
# R3. Reviewer calibration: identical-pixel controls and reversed presentation order.
_rz=w_start(R_SEEDS[0])
with torch.no_grad():
 _,_rt=w_forward(_rz,50,W_NEUTRAL);_ra,_rm=d_properties(_rt['cross_maps'][D_HAND].mean(0))
R_SOURCE_BOX=r_context_box(_rm)
R_CAL_IMAGE=Image.open(R_OLD_OUT/'s123200100_r09_committed.png').convert('RGB')
R_CAL={}
for tag,a,b in [('identity_source',D_SOURCE,D_SOURCE.copy()),('identity_final',R_CAL_IMAGE,R_CAL_IMAGE.copy()),('order_AB',D_SOURCE,R_CAL_IMAGE),('order_BA',R_CAL_IMAGE,D_SOURCE)]:
 R_CAL[tag]=r_pair(a,b,R_SOURCE_BOX,'cal_'+tag)
 print(tag, R_CAL[tag]['parsed'] or R_CAL[tag]['raw'],flush=True)
R_CAL_SUMMARY={'identity_source_pass':R_CAL['identity_source']['valid_JSON'] and R_CAL['identity_source']['parsed'].get('difference')=='none' and R_CAL['identity_source']['parsed'].get('preference')=='tie','identity_final_pass':R_CAL['identity_final']['valid_JSON'] and R_CAL['identity_final']['parsed'].get('difference')=='none' and R_CAL['identity_final']['parsed'].get('preference')=='tie','all_valid_JSON':all(v['valid_JSON'] for v in R_CAL.values()),'limits':'Two identity controls and one reversed-order pair are a small sanity test; they do not validate anatomical accuracy.'}
_ab=(R_CAL['order_AB']['parsed'] or {}).get('preference');_ba=(R_CAL['order_BA']['parsed'] or {}).get('preference')
R_CAL_SUMMARY['preference_order_consistent']=_ab is not None and _ba=={'A':'B','B':'A','tie':'tie','uncertain':'uncertain'}.get(_ab)
R_CAL_SUMMARY['raw_preference_AB']=_ab;R_CAL_SUMMARY['raw_preference_BA']=_ba
r_dump('reviewer_calibration.json',R_CAL);r_dump('reviewer_calibration_summary.json',R_CAL_SUMMARY)
print('CALIBRATION SUMMARY',R_CAL_SUMMARY)


In [ ]:
# R4. Sequential semantic hypothesis -> joint controls -> evidence-based commitment.
R_HYPOTHESIS_PROMPT='''Build a testable account of the visible hand using the supplied whole-image and enlarged-context observations, the observer sanity results, and prior intervention evidence. Use anatomical and image-formation knowledge: contours should connect coherently to the palm, joint bends and relative proportions must allow the pose, and apparent missing boundaries can be valid occlusion or perspective. Identify the most supported relationship to examine, a competing explanation, what to preserve, and a visible consequence that would distinguish them. Unresolved is a valid hypothesis, but can support an information-gathering probe. Do not infer a finger identity from a head or an attention map. Return the requested typed choices. These choices are passed explicitly to the next call; you are not selecting numerical controls in this call.'''
R_PROPOSAL_PROMPT='''The selected_hypothesis field is the OUTPUT OF A PREVIOUS COMPLETED Jev call. Use it explicitly when choosing each component of the next joint experiment. The complete compact ledger includes accepted and rejected configurations, measured responses and observed side effects; inspect those before choosing. Coordinate which visible relationship the edit tests, what must stay stable, and what already-tested result it should improve upon or distinguish. Avoid repeating an inconclusive configuration without a stated reason tied to a changed state or a longer observation horizon. You may make several bounded edits together, or preserve settings. Candidate control values are mechanisms, not finger-size labels. A new configuration is a probe, not a claim of improvement. The executor will test the joint proposal and a half-strength version against continued current settings, from exactly the same latent and noise schedule. Each parameter question reads this same completed hypothesis and history, but does not read other parameter answers in this batch.'''
R_SELECT_PROMPT='''Choose from actual matched candidate trajectories. Each started from the same checkpoint and was observed after two and up to six steps. You have paired whole-image/context observations, exact changes, signed property errors, and the previous semantic hypothesis. Ask whether a specific predicted relationship changed coherently, whether that consequence persisted, and whether neighboring structure or the background suffered. The visual reviewer is evidence with measured limitations, not an oracle; a failed sanity test limits any conclusion that depends on it. A forecast is not a final image. Do not select an edit because it is stronger, novel, sharper, or has a lower internal loss. A supported beneficial edit may be committed; if none is distinguishable, retaining the current configuration is valid. Record whether the hypothesis gained, lost, or still lacks support. The executor commits the selected branch's first five steps and returns to the next round; it cannot stop early.'''
R_HYPOTHESES=copy.deepcopy(W_HYPOTHESES)
R_COMPETING={'overlap':'A contour is hidden by valid overlap; changing geometry could damage the pose.','perspective':'Apparent width or length reflects foreshortening rather than malformed anatomy.','geometry':'The attachment, articulation or relative proportions themselves may be inconsistent.','rendering':'Underlying structure may be plausible but boundaries, shading or material cues are ambiguous.','observer':'The current visual description may be inaccurate; require a distinguishing observation.','none':'No supported defect or competing account can yet be identified.'}
R_PRESERVE={'pose':'Preserve pose and silhouette while testing interior continuity.','neighbors':'Preserve nearby finger/palm relationships while examining the uncertain feature.','surface':'Preserve material and lighting while testing contour/attachment structure.','global':'Preserve global composition, sleeve and background.','all':'No supported structural change: preserve current content while investigating.'}
R_HQ={'relationship':w_choice(R_HYPOTHESIS_PROMPT,R_HYPOTHESES),'alternative':w_choice('Using the supplied observations and the complete intervention ledger, which alternative explanation most needs to be distinguished?',R_COMPETING),'preserve':w_choice('Which preserved relationship is most relevant to this proposed investigation?',R_PRESERVE),'prediction':w_choice('Which observable consequence would distinguish a useful test under this evidence? A prediction is not a reported improvement.',W_EXPECT)}
R_REQUESTS=[]
def r_call(tag,state,questions):
 assert R_COUNTS['jev_calls']<R_CONFIG['max_requests_including_tests_and_repeated_probe_responses']
 R_COUNTS['jev_calls']+=1;serial=R_COUNTS['jev_calls'];t=time.time()
 payload={'model':'jev-1.13.0','state':state,'questions':questions};r_dump(f'api_{serial:03}_{tag}_request.json',payload)
 for attempt in range(3):
  response=requests.post('https://api.typesafe.ai/v1/systemone',headers={'Authorization':'Bearer '+_d_key},json=payload,timeout=90)
  if response.status_code==200:break
  r_dump(f'api_{serial:03}_attempt{attempt}.json',{'status':response.status_code})
  if response.status_code not in [429,500,502,503,504,529]:raise RuntimeError(f'Jev HTTP {response.status_code}; request saved; body withheld')
  time.sleep(2**attempt)
 if response.status_code!=200:raise RuntimeError('Jev transport retries exhausted')
 raw=response.json();r_dump(f'api_{serial:03}_{tag}_response.json',raw);result={}
 for key,q in questions.items():
  v=raw['answers'][key]['choice'];assert v in q['criteria'],f'Unsupported choice: {key}'
  result[key]=v
 R_COUNTS['jev_seconds']+=time.time()-t;R_REQUESTS.append({'tag':tag,'state':copy.deepcopy(state),'answers':result,'serial':serial})
 return result

def r_hypothesis_record(answers,tag):
 return {'source_call':tag,'choices':answers,'relationship':R_HYPOTHESES[answers['relationship']],'alternative':R_COMPETING[answers['alternative']],'preserve':R_PRESERVE[answers['preserve']],'predicted_observation':W_EXPECT[answers['prediction']]}
def r_pquestions(p):
 q,values=w_proposal_questions(p)
 for k in ['hypothesis','expected','counterevidence']:q.pop(k)
 q['experiment_reason']=w_choice(R_PROPOSAL_PROMPT,{'new_mechanism':'Test a configuration/response not resolved by existing probes.','state_change':'Retest because the current state or visible relationship materially changed.','longer_horizon':'Retest a previously short probe with the now-longer observation horizon.','preserve':'No useful distinguishable edit is supported; preserve settings.'})
 return q,values
r_dump('controller_prompts.json',{'hypothesis':R_HYPOTHESIS_PROMPT,'proposal':R_PROPOSAL_PROMPT,'selection':R_SELECT_PROMPT,'hypothesis_questions':R_HQ})
for name,prompt in [('Hypothesis call',R_HYPOTHESIS_PROMPT),('Conditioned control call',R_PROPOSAL_PROMPT),('Selection call',R_SELECT_PROMPT)]:display(Markdown('### '+name+'\n'+prompt))


In [ ]:
# R5. Fix the reviewer layout after its failed four-panel calibration; retain those failures.
R_PAIR_PROMPT_V2='''Compare IMAGE A and IMAGE B. They have the same framing and scale. These are two separate candidate renderings, not two zoom levels. They may also be exact duplicates. Look at matching locations before deciding. Return a compact JSON object: {"difference":"none|visible|uncertain", "preference":"A|B|tie|uncertain", "observations":["up to three specific differences or an explicit statement of no visible difference"], "collateral":"unintended differences, if any", "uncertainty":"what you cannot establish"}. Use difference=none and preference=tie if no differences are visible. Prefer an image only for a concrete improvement in coherent shape, attachment, articulation, occlusion or material continuity; do not equate sharpness with correct anatomy. A visible difference can have an uncertain preference. Do not repeat these instructions.'''
R_VISION_CACHE_V2={}
@torch.inference_mode()
def r_vision_two(a,b,prompt,tag):
 key=hashlib.sha256(a.tobytes()+b.tobytes()+prompt.encode()).hexdigest()
 if key in R_VISION_CACHE_V2:return copy.deepcopy(R_VISION_CACHE_V2[key])
 a.save(R_OUT/f'{tag}_A.png');b.save(R_OUT/f'{tag}_B.png');t=time.time();R_COUNTS['vision_calls']+=1
 content=[{'type':'text','text':'IMAGE A:'},{'type':'image'},{'type':'text','text':'IMAGE B:'},{'type':'image'},{'type':'text','text':prompt}]
 template=R_VP.apply_chat_template([{'role':'user','content':content}],tokenize=False,add_generation_prompt=True)
 inp=R_VP(text=[template],images=[a,b],padding=True,return_tensors='pt').to('cuda')
 assert inp.image_grid_thw.shape[0]==2,'Reviewer must receive two distinct image inputs.'
 out=W_VM.generate(**inp,max_new_tokens=230,do_sample=False,temperature=None,top_p=None,top_k=None)
 raw=R_VP.batch_decode(out[:,inp.input_ids.shape[1]:],skip_special_tokens=True,clean_up_tokenization_spaces=False)[0]
 try:parsed=json.loads(raw[raw.index('{'):raw.rindex('}')+1])
 except (ValueError,json.JSONDecodeError):parsed=None
 valid=isinstance(parsed,dict) and parsed.get('difference') in ['none','visible','uncertain'] and parsed.get('preference') in ['A','B','tie','uncertain'] and isinstance(parsed.get('observations'),list) and len(parsed['observations'])<=3
 result={'raw':raw,'parsed':parsed,'valid_schema':valid,'source':'Qwen2-VL-2B separate-image comparison','image_A':f'{tag}_A.png','image_B':f'{tag}_B.png','exact_pixel_identity':bool(np.array_equal(np.asarray(a),np.asarray(b))),'pixel_MAE':float(np.abs(np.asarray(a).astype('float32')-np.asarray(b).astype('float32')).mean()/255),'seconds':time.time()-t}
 R_COUNTS['vision_seconds']+=result['seconds'];R_VISION_CACHE_V2[key]=copy.deepcopy(result);r_dump(f'{tag}_vision.json',result);return result
R_CAL_V2={}
for tag,a,b in [('identity_source',D_SOURCE,D_SOURCE.copy()),('identity_final',R_CAL_IMAGE,R_CAL_IMAGE.copy()),('order_AB',D_SOURCE,R_CAL_IMAGE),('order_BA',R_CAL_IMAGE,D_SOURCE)]:
 R_CAL_V2[tag]=r_vision_two(a,b,R_PAIR_PROMPT_V2,'cal2_'+tag)
 print(tag,R_CAL_V2[tag]['parsed'] or R_CAL_V2[tag]['raw'],flush=True)
R_CAL_V2_SUMMARY={'identity_source_pass':False,'identity_final_pass':False,'all_valid_schema':all(x['valid_schema'] for x in R_CAL_V2.values())}
for tag in ['identity_source','identity_final']:
 p=R_CAL_V2[tag]['parsed'] or {};R_CAL_V2_SUMMARY[tag+'_pass']=p.get('difference')=='none' and p.get('preference')=='tie'
_ab=(R_CAL_V2['order_AB']['parsed'] or {}).get('preference');_ba=(R_CAL_V2['order_BA']['parsed'] or {}).get('preference')
R_CAL_V2_SUMMARY['preference_order_consistent']=_ab is not None and _ba=={'A':'B','B':'A','tie':'tie','uncertain':'uncertain'}.get(_ab)
R_CAL_V2_SUMMARY['real_change_detected_both_orders']=all((R_CAL_V2[k]['parsed'] or {}).get('difference')=='visible' for k in ['order_AB','order_BA'])
R_CAL_V2_SUMMARY['limits']='Two duplicate controls and one unlabelled changed pair; not a validated anatomical evaluator.'
r_dump('reviewer_calibration_v2.json',R_CAL_V2);r_dump('reviewer_calibration_v2_summary.json',R_CAL_V2_SUMMARY)
r_dump('reviewer_prompt_v2.json',{'prompt':R_PAIR_PROMPT_V2,'separate_image_inputs':True})
print('REVISED CALIBRATION',R_CAL_V2_SUMMARY)


In [ ]:
# R6. Complete intervention memory: rejected trials are first-class evidence.
R_PARAMS={}
def r_param_id(p):
 pid=hashlib.sha256(json.dumps(p,sort_keys=True).encode()).hexdigest()[:12]
 R_PARAMS[pid]=copy.deepcopy(p);return pid
R_PRIOR_GROUPS={}
for old in sorted(R_OLD_OUT.glob('s*_r*_decision.json')):
 rec=json.loads(old.read_text())
 for c in rec['candidates']:
  pid=r_param_id(c['parameters']);group=R_PRIOR_GROUPS.setdefault(pid,{'configuration_id':pid,'trials':0,'retained':0,'RGB_MAE':[],'indices':[]})
  group['trials']+=1;group['retained']+=int(c['id']==rec['selected_id']);group['RGB_MAE'].append(c['matched_change']['RGB_MAE']);group['indices'].append(rec['index'])
R_PRIOR=[]
for g in R_PRIOR_GROUPS.values():
 R_PRIOR.append({'configuration_id':g['configuration_id'],'trials':g['trials'],'retained':g['retained'],'RGB_MAE_range':[min(g['RGB_MAE']),max(g['RGB_MAE'])],'schedule_range':[min(g['indices']),max(g['indices'])],'quality':'Previous run unresolved; this describes tested effects, not good/bad anatomy.','horizon':2})

def r_memory(seed,ledger=None):
 records=[x for x in (R_LEDGER if ledger is None else ledger) if x['seed']==seed]
 index=[]
 for x in records:
  index.append({k:x[k] for k in ['trial_id','configuration_id','index','horizon','accepted','RGB_MAE_early','RGB_MAE_late','visual_preference','difference','repeat_of']})
 relevant={x['configuration_id'] for x in records}|{x['configuration_id'] for x in R_PRIOR}
 return {'prior_experiment_configurations':R_PRIOR,'configuration_catalogue':{k:R_PARAMS[k] for k in sorted(relevant)},'all_current_seed_trials':index,'recent_detailed_trials':records[-6:],'total_current_seed_trials':len(records),'rejected_current_seed_trials':sum(not x['accepted'] for x in records),'interpretation':'Acceptance is not ground-truth quality; rejected probes still constrain hypotheses. Different noise levels are not identical experimental conditions.'}

def r_repeats(seed,p):
 pid=r_param_id(p)
 return [x['trial_id'] for x in R_LEDGER if x['seed']==seed and x['configuration_id']==pid]

# Meaningful regression test: a rejected branch with a distinctive observation must survive packet construction.
_test_pid=r_param_id(W_NEUTRAL)
_test_record={'seed':-1,'trial_id':'rejected-memory-regression','configuration_id':_test_pid,'index':50,'horizon':6,'accepted':False,'RGB_MAE_early':.01,'RGB_MAE_late':.02,'visual_preference':'uncertain','difference':'visible','repeat_of':[],'observed_consequence':'Specific rejected branch evidence retained.'}
_test_mem=r_memory(-1,[_test_record])
assert _test_mem['rejected_current_seed_trials']==1
assert _test_mem['all_current_seed_trials'][0]['trial_id']=='rejected-memory-regression'
assert _test_mem['recent_detailed_trials'][0]['observed_consequence']=='Specific rejected branch evidence retained.'
assert _test_mem['configuration_catalogue'][_test_pid]==W_NEUTRAL
R_TESTS={'rejected_trial_memory':True,'previous_artifacts_unchanged':hashlib.sha256((R_OLD_OUT/'protocol.json').read_bytes()).hexdigest()==R_OLD_MANIFEST_HASH}
r_dump('regression_tests.json',R_TESTS)
print('Memory regression passed. Prior accepted and rejected configurations:',len(R_PRIOR))
# Also choose layer in the prior semantic call, so head IDs in the control call are unambiguous.
R_HQ['layer']=w_choice('Which of the two available self-attention layers is the most useful next probe given its measured responses and the current image relationship? This answer is passed to the separate parameter call.',{'8x8':W_LAYERS['8x8'],'16x16':W_LAYERS['16x16']})
print('Hypothesis and layer are now both completed before selecting head parameters.')


In [ ]:
# R7. Bound the pilot after reviewer failure; verify distinct pixels really reach the model.
R_CONFIG.update(rounds_per_seed=2,decision_indices=[50,70],planned_controller_calls=12,max_requests_including_tests_and_repeated_probe_responses=24,reason_for_smaller_pilot='Reviewer failed identity/order/change checks; test plumbing and four real decisions, not a long quality search.',review_views='Separate image inputs; whole-image and same-coordinate enlarged context reviewed in separate passes')
r_dump('protocol.json',R_CONFIG)
_test_template=R_VP.apply_chat_template([{'role':'user','content':[{'type':'image'},{'type':'image'},{'type':'text','text':'Compare.'}]}],tokenize=False,add_generation_prompt=True)
_tinp=R_VP(text=[_test_template],images=[D_SOURCE,R_CAL_IMAGE],padding=True,return_tensors='pt')
_grid=_tinp.image_grid_thw.cpu();_patches=[int(x.prod()) for x in _grid]
assert len(_patches)==2 and _patches[0]==_patches[1]
_n=_patches[0];_patchdiff=float((_tinp.pixel_values[:_n]-_tinp.pixel_values[_n:2*_n]).abs().mean())
assert _patchdiff>0
R_TESTS['different_images_reach_reviewer']=True;R_TESTS['reviewer_input_patch_mean_difference']=_patchdiff
print('Separate-image tensor check passed:',_patchdiff)

def r_visual_evidence(result):
 return {'parsed':result.get('parsed'),'valid_schema':result.get('valid_schema',result.get('valid_JSON',False)),'raw_if_invalid':result.get('raw') if not result.get('valid_schema',result.get('valid_JSON',False)) else None,'exact_pixel_identity':result.get('exact_pixel_identity'),'pixel_MAE':result.get('pixel_MAE'),'source':result.get('source'),'interpretation':'Unvalidated visual report. Calibration failed; it cannot alone establish image-quality improvement.'}
def r_pair_views(a,b,box,tag):
 whole=r_vision_two(a,b,R_PAIR_PROMPT_V2,tag+'_whole')
 context=r_vision_two(a.crop(box).resize((512,512)),b.crop(box).resize((512,512)),R_PAIR_PROMPT_V2,tag+'_context')
 return {'whole':r_visual_evidence(whole),'context':r_visual_evidence(context),'crop_xyxy':list(box),'same_coordinates':True}

# A narrow prompt for a single actual image avoids the earlier four-panel confusion.
R_CURRENT_PROMPT='''Describe concrete visible relationships in this one image of a hand: where the thumb emerges relative to the palm, continuity of the finger-to-palm contours, relative finger widths/lengths allowing perspective, and any ambiguous overlap. Do not just say it is an open hand. State which relationships are unclear and offer a plausible perspective/occlusion explanation when applicable. Do not invent hidden anatomy. Return a concise paragraph, not a score or a tensor recommendation.'''
@torch.inference_mode()
def r_current(im,box,tag):
 # Existing one-image transport preserves the raw description even when it is not JSON.
 reports=[]
 for name,view in [('whole',im),('context',im.crop(box).resize((512,512)))]:
  res=r_vision(view,R_CURRENT_PROMPT,tag+'_'+name)
  reports.append({'view':name,'description':res['raw'],'view_file':res['view_file']})
 return {'views':reports,'crop_xyxy':list(box),'reviewer_calibration':R_CAL_V2_SUMMARY}
R_HQ['layer']['instructions']+=' The selected layer will be fixed before the subsequent head-parameter questions.'
r_dump('regression_tests.json',R_TESTS)
print('Four decision cycles planned, then complete both 50-step trajectories. No further Jev calls to repeat prior calibration.')


In [ ]:
# R8. Matched six-step probes, complete rejected-branch ledger, and live reporting.
R_PROGRESS=[];R_PILOT_START=None

def r_probe(z,i,p,anchor,tag):
 zz=z.clone();records=[];states=[];clean_by_step={};images={}
 for j in range(min(6,100-i)):
  eps,clean,info,mask=w_eval(zz,i+j,p,anchor);zz=w_step(zz,eps,i+j)
  states.append(zz.detach().cpu().clone());records.append(info)
  if j+1 in [2,5,6] or i+j==99:
   clean_by_step[j+1]=clean.detach().cpu();images[j+1]=d_decode(zz if i+j==99 else clean)
   images[j+1].save(R_OUT/f'{tag}_step{j+1}.png')
 torch.save({'states_after_steps':states,'clean_forecasts':clean_by_step,'parameters':p,'measurements':records,'starting_index':i},R_OUT/f'{tag}_trace.pt')
 return {'p':copy.deepcopy(p),'states':states,'measurements':records,'images':images,'tag':tag,'horizon':len(states)}

def r_numeric(candidate,reference,mask,step):
 c=candidate['images'][step];b=reference['images'][step]
 return {**w_image_delta(c,b,mask),'epsilon_effect_rms':candidate['measurements'][step-1]['delta_eps_rms'],'clean_effect_rms':candidate['measurements'][step-1]['clean_effect_rms'],'signed_property_error':candidate['measurements'][step-1]['signed_error']}

def r_render():
 parts=['<!doctype html><html><head><meta charset="utf-8"><meta http-equiv="refresh" content="20"><title>Jev feedback repairs pilot</title><style>body{font:15px system-ui;background:#171a20;color:white;margin:24px}.row{display:flex;gap:12px;flex-wrap:wrap}figure{margin:0}img{max-width:256px}pre{white-space:pre-wrap}article{border:1px solid #555;padding:12px;margin:16px 0}</style></head><body><h1>Feedback repairs: tested pilot</h1>',f'<p>Completed decisions {len(R_ROUNDS)}/4; completed trajectories {len(R_FINALS)}/2; Jev calls {R_COUNTS["jev_calls"]}; running {R_RUNNING}</p><p>Reviewer calibration failed. Images and observations are retained; acceptance is not proof of quality.</p>',f'<figure><img src="{w_thumb(D_SOURCE,256)}"><figcaption>Original fixed seed-123 source</figcaption></figure>']
 for seed in R_SEEDS:
  parts.append(f'<h2>Noise seed {seed}</h2><div class="row">')
  for f in R_PROGRESS:
   if f['seed']==seed:parts.append(f'<figure><img src="{w_thumb(Image.open(R_OUT/f["image"]))}"><figcaption>After schedule index {f["index"]}</figcaption></figure>')
  parts.append('</div>')
 for r in R_ROUNDS:
  parts.append(f'<article><h2>Seed {r["seed"]}, decision at {r["index"]}: {r["selection"]["candidate"]}</h2><p>Hypothesis: {html.escape(r["hypothesis"]["choices"]["relationship"])}, selected {r["selected_kind"]}</p><div class="row">')
  for c in r['candidates']:parts.append(f'<figure><img src="{w_thumb(Image.open(R_OUT/c["late_image"]))}"><figcaption>{c["id"]}: {c["kind"]}</figcaption></figure>')
  parts.append('</div><details><summary>Every setting, observation, repeat flag and decision</summary><pre>'+html.escape(json.dumps(r,indent=2))+'</pre></details></article>')
 parts.append('<h2>Reviewer calibration</h2><pre>'+html.escape(json.dumps(R_CAL_V2_SUMMARY,indent=2))+'</pre>')
 if R_FAILURES:parts.append('<h2>Failures</h2><pre>'+html.escape(json.dumps(R_FAILURES,indent=2))+'</pre>')
 parts.append('</body></html>');(R_OUT/'live.html').write_text(''.join(parts))

def r_decide(z,i,p,anchor,seed):
 tag=f's{seed}_i{i:03}'
 eps,clean,info,mask=w_eval(z,i,p,anchor);im=d_decode(clean);box=r_context_box(mask)
 im.save(R_OUT/f'{tag}_before.png');current_review=r_current(im,box,tag+'_current')
 memory=r_memory(seed)
 base_state={'architecture':W_ARCH,'original_prompt':D_PROMPT,'current_index':i,'current_configuration':p,'current_views':current_review,'current_internal':w_small(info),'memory':memory,'observer_calibration':R_CAL_V2_SUMMARY,'scope':'Frozen SD1.5; same original source; independent reviewer sanity tests failed, so anatomical conclusions require caution.'}
 hypothesis_answers=r_call(tag+'_hypothesis',dict(base_state,protocol=R_HYPOTHESIS_PROMPT),R_HQ)
 hypothesis=r_hypothesis_record(hypothesis_answers,tag+'_hypothesis')
 # Completed semantic inference is now literal input to all parameter questions.
 basis=copy.deepcopy(p);basis['layer']=hypothesis_answers['layer']
 if basis['layer']!=p['layer']:basis['gates']=[1.]*8;basis['u']=[0.]*8
 qs,vals=r_pquestions(basis);qs.pop('layer');vals['layer']={'fixed':basis['layer']}
 for h in range(8):
  for k in ['gates','u']:
   qs[f'{k}_{h}']['instructions']=qs[f'{k}_{h}']['instructions'].split('Layer choice is independent')[0]+' The layer is already fixed to '+basis['layer']+'. Use selected_hypothesis and the ledger, including rejected outcomes.'
 proposal_state=dict(base_state,protocol=R_PROPOSAL_PROMPT,selected_hypothesis=hypothesis,selected_layer=basis['layer'],parameter_start=basis)
 a=r_call(tag+'_controls',proposal_state,qs);proposed=w_decode(basis,dict(a,layer='fixed'),vals)
 R_TESTS['hypothesis_forwarded_exactly']=R_REQUESTS[-1]['state']['selected_hypothesis']==hypothesis
 repeat_ids=r_repeats(seed,proposed)
 # A repeated parameter vector is recorded explicitly. No random replacement or forced acceptance.
 if repeat_ids:
  follow=dict(proposal_state,previous_proposal=proposed,repeated_trial_ids=repeat_ids,repeat_instruction='This proposal repeats the listed previous configurations. Choose whether changed state or the declared horizon justifies it, or preserve current settings. Do not invent a new measurement.')
  response=r_call(tag+'_repeat_check',follow,{'repeat_action':w_choice('Given the full ledger and current hypothesis, is this repeated probe still informative?',{'state_changed':'Repeat: current state or relationship meaningfully differs.','horizon_changed':'Repeat: the prior observation horizon was shorter.','hold':'Do not spend another probe on this configuration; continue current parameters.'})})
  if response['repeat_action']=='hold':proposed=copy.deepcopy(p)
  repeat_decision=response
 else:repeat_decision={'repeat_action':'new_configuration'}
 configurations=[('continue',copy.deepcopy(p)),('joint',proposed),('halfway',w_half(p,proposed))]
 random.Random(seed+i).shuffle(configurations);branches=[];cache={}
 for j,(kind,cp) in enumerate(configurations):
  pid=r_param_id(cp)
  if pid not in cache:cache[pid]=r_probe(z,i,cp,anchor,f'{tag}_branch{j}')
  branch=dict(cache[pid],id=f'candidate_{j}',kind=kind,configuration_id=pid);branches.append(branch)
 reference=next(b for b in branches if b['kind']=='continue');horizon=reference['horizon'];evidence=[];pending=[]
 for b in branches:
  early=r_numeric(b,reference,mask,2);late=r_numeric(b,reference,mask,horizon)
  # Candidate labels are deterministically shuffled; reviewed A/B order is separately randomized.
  swapped=bool(random.Random(seed+i+int(b['id'].split('_')[-1])).getrandbits(1))
  if late['identical_pixels'] and early['identical_pixels']:
   reviews={'early':{'exact_pixel_identity':True},'late':{'exact_pixel_identity':True},'A_identity':b['id'] if swapped else reference['id'],'B_identity':reference['id'] if swapped else b['id']}
  else:
   rev={}
   for name,step in [('early',2),('late',horizon)]:
    aa,bb=(b['images'][step],reference['images'][step]) if swapped else (reference['images'][step],b['images'][step])
    rev[name]=r_pair_views(aa,bb,box,f'{tag}_{b["id"]}_{name}')
   reviews={**rev,'A_identity':b['id'] if swapped else reference['id'],'B_identity':reference['id'] if swapped else b['id']}
  tid=f'{tag}_{b["id"]}';lateimage=f'{b["tag"]}_step{horizon}.png'
  ev={'id':b['id'],'kind':b['kind'],'configuration_id':b['configuration_id'],'parameters':b['p'],'early':early,'late':late,'paired_reviews':reviews,'horizon':horizon,'late_image':lateimage,'internal_early':w_small(b['measurements'][1]),'internal_late':w_small(b['measurements'][-1])};evidence.append(ev)
  pr=reviews.get('late',{}).get('whole',{}).get('parsed') or {}
  pending.append({'seed':seed,'trial_id':tid,'configuration_id':b['configuration_id'],'index':i,'horizon':horizon,'accepted':False,'RGB_MAE_early':early['RGB_MAE'],'RGB_MAE_late':late['RGB_MAE'],'visual_preference':pr.get('preference','tie' if late['identical_pixels'] else 'unavailable'),'difference':pr.get('difference','none' if late['identical_pixels'] else 'unavailable'),'repeat_of':r_repeats(seed,b['p']),'observed_consequence':reviews,'signed_error_early':early['signed_property_error'],'signed_error_late':late['signed_property_error'],'archive':f'{tag}_decision.json'})
 selection_state={'protocol':R_SELECT_PROMPT,'selected_hypothesis':hypothesis,'observer_calibration':R_CAL_V2_SUMMARY,'current_configuration_candidate':reference['id'],'matched_candidates':[{k:v for k,v in e.items() if k not in ['kind','late_image']} for e in evidence],'memory':r_memory(seed),'configuration_repeat_check':repeat_decision}
 selection=r_call(tag+'_select',selection_state,{'candidate':w_choice(R_SELECT_PROMPT,{b['id']:'Commit this candidate prefix; assess its actual matched evidence.' for b in branches}),'hypothesis_result':w_choice('What happened to the previously stated hypothesis?',{'supported':'Specific predicted relationship is supported by observed changes.','contradicted':'The expected relationship or preservation prediction was contradicted.','unresolved':'Evidence does not distinguish the competing explanations.'}),'quality':w_choice('What relative quality conclusion follows from the actual paired/context observations and calibration?',{'better':'A concrete supported improvement without material collateral damage.','worse':'Concrete supported deterioration.','unresolved':'No defensible quality ordering.'})})
 chosen=next(b for b in branches if b['id']==selection['candidate'])
 for x in pending:x['accepted']=x['trial_id']==f'{tag}_{chosen["id"]}'
 R_LEDGER.extend(pending)
 record={'seed':seed,'index':i,'hypothesis':hypothesis,'proposal_answers':a,'proposed_configuration':proposed,'repeat_ids':repeat_ids,'repeat_decision':repeat_decision,'selection':selection,'selected_kind':chosen['kind'],'committed_configuration':chosen['p'],'candidates':evidence,'memory_before_trial_count':memory['total_current_seed_trials'],'memory_after_trial_count':r_memory(seed)['total_current_seed_trials'],'hypothesis_forwarding_verified':R_TESTS['hypothesis_forwarded_exactly']}
 R_ROUNDS.append(record);r_dump(f'{tag}_decision.json',record);r_dump('all_trial_ledger.json',R_LEDGER)
 assert r_memory(seed)['total_current_seed_trials']==sum(x['seed']==seed for x in R_LEDGER)
 torch.save({'latent':chosen['states'][4],'index_next':i+5,'parameters':chosen['p']},R_OUT/f'{tag}_commit.pt')
 print(f'Seed {seed} index {i}: hypothesis={hypothesis_answers["relationship"]}, chose={chosen["kind"]}, quality={selection["quality"]}; ledger={r_memory(seed)["total_current_seed_trials"]} trials',flush=True)
 r_render()
 return chosen['states'][4].to('cuda'),copy.deepcopy(chosen['p']),chosen['images'][5]

r_render();display(HTML(f'<a target="_blank" href="/files/workspace/crazy_exp/{R_OUT}/live.html">Watch the repaired-feedback pilot</a>'))
print('Ready: four hypothesis-conditioned decisions, six-step probes, all accepted/rejected trials retained.')


In [ ]:
# R9. Reproducible execution checks and the bounded two-seed diffusion test.
assert R_TESTS['rejected_trial_memory'] and R_TESTS['different_images_reach_reviewer']
assert not R_ROUNDS and not R_RUNNING,'Do not launch twice; use saved checkpoints for recovery.'
# Save the actual runtime definitions used here for audit without executing old runs.
_rsource='\n\n'.join(s for s in get_ipython().history_manager.input_hist_raw if re.match(r'^# R\d+',s))
(R_OUT/'executed_cells.py').write_text(_rsource)
(R_OUT/'parent_attention_processor.py').write_text((R_OLD_OUT/'attention_processor.py').read_text())
r_dump('controller_prompts_final.json',{'hypothesis':R_HYPOTHESIS_PROMPT,'proposal':R_PROPOSAL_PROMPT,'selection':R_SELECT_PROMPT,'hypothesis_questions':R_HQ,'review_pair':R_PAIR_PROMPT_V2,'review_current':R_CURRENT_PROMPT})
R_RUNNING=True;R_PILOT_START=time.time();r_render()
try:
 for seed in R_SEEDS:
  print('START seed',seed,flush=True);z=w_start(seed);p=copy.deepcopy(W_NEUTRAL)
  with torch.no_grad():
   _,tt=w_forward(z,50,W_NEUTRAL);anchor,_=d_properties(tt['cross_maps'][D_HAND].mean(0));anchor=anchor.detach()
  i=50
  while i<100:
   if i in R_CONFIG['decision_indices']:
    z,p,im=r_decide(z,i,p,anchor,seed);i+=5
   else:
    eps,clean,info,mask=w_eval(z,i,p,anchor);z=w_step(z,eps,i);i+=1
    torch.save({'latent_after':z.cpu(),'index_next':i,'parameters':p,'measurements':info},R_OUT/f's{seed}_ordinary_advance{i:03}.pt')
    if i%5!=0:continue
    im=d_decode(z if i==100 else clean)
   name=f's{seed}_progress_{i:03}.png';im.save(R_OUT/name)
   R_PROGRESS.append({'seed':seed,'index':i,'image':name});r_render()
  final=d_decode(z);filename=f's{seed}_final.png';final.save(R_OUT/filename)
  torch.save(z.cpu(),R_OUT/f's{seed}_final_latent.pt');R_FINALS[seed]=filename;r_dump('finals.json',R_FINALS)
  print('FINISHED seed',seed,flush=True)
 print('PILOT COMPLETE',len(R_ROUNDS),'decisions;',len(R_FINALS),'trajectories;',R_COUNTS['jev_calls'],'Jev calls.',flush=True)
except Exception as exc:
 failure={'type':type(exc).__name__,'message':str(exc),'traceback':traceback.format_exc(),'decisions_completed':len(R_ROUNDS)}
 R_FAILURES.append(failure);r_dump('failure.json',failure);print('PILOT ERROR:',type(exc).__name__,str(exc),flush=True);raise
finally:
 R_RUNNING=False
 r_dump('status.json',{'complete':len(R_FINALS)==2,'rounds':len(R_ROUNDS),'finals':R_FINALS,'counts':R_COUNTS,'pilot_wall_seconds':time.time()-R_PILOT_START,'diffusion_counters_since_setup':{k:W_COUNTS[k]-R_PRE_FORWARD_COUNTS[k] for k in ['full_forwards','partial_gradient_forwards','backwards']}})
 r_render()


In [ ]:
# R10. Post-run verification of the actual handoffs, full memory, probe horizons and committed prefixes.
assert len(R_ROUNDS)==4 and len(R_FINALS)==2 and not R_RUNNING,'Pilot incomplete; inspect failure.json before continuing.'
by_tag={x['tag']:x for x in R_REQUESTS}
checks=[]
for r in R_ROUNDS:
 tag=f's{r["seed"]}_i{r["index"]:03}'
 h=by_tag[tag+'_hypothesis'];c=by_tag[tag+'_controls']
 before=[x for x in R_LEDGER if x['seed']==r['seed'] and x['index']<r['index']]
 carried=c['state']['memory']['all_current_seed_trials'];carried_ids={x['trial_id'] for x in carried}
 expected_ids={x['trial_id'] for x in before}
 chosen=next(x for x in r['candidates'] if x['id']==r['selection']['candidate'])
 trace_name=chosen['late_image'].replace('_step6.png','_trace.pt')
 trace=torch.load(R_OUT/trace_name,map_location='cpu',weights_only=False)
 commit=torch.load(R_OUT/f'{tag}_commit.pt',map_location='cpu',weights_only=False)
 checks.append({'tag':tag,'hypothesis_completed_before_controls':h['serial']<c['serial'],'exact_hypothesis_forwarded':c['state']['selected_hypothesis']['choices']==h['answers'],'layer_fixed_before_head_questions':c['state']['selected_layer']==h['answers']['layer'],'all_prior_seed_trials_carried':carried_ids==expected_ids,'rejected_trials_in_context':sum(not x['accepted'] for x in carried),'six_step_horizon':all(x['horizon']==6 for x in r['candidates']),'only_five_steps_committed':commit['index_next']==r['index']+5,'committed_prefix_exact':bool(torch.equal(commit['latent'],trace['states_after_steps'][4]))})
for check in checks:
 for k,v in check.items():
  if isinstance(v,bool):assert v,(check['tag'],k)
for seed in R_SEEDS:assert [x['index'] for x in R_PROGRESS if x['seed']==seed]==list(range(55,101,5))
R_TESTS.update(actual_controller_checks=checks,all_fifty_steps_per_seed=True,all_twelve_trial_records_retained=len(R_LEDGER)==12,all_controller_checks_pass=True)
assert R_TESTS['all_twelve_trial_records_retained']
R_TESTS['old_protocol_unchanged']=hashlib.sha256((R_OLD_OUT/'protocol.json').read_bytes()).hexdigest()==R_OLD_MANIFEST_HASH
r_dump('regression_tests.json',R_TESTS)
rows=[]
for r in R_ROUNDS:
 ref=next(x for x in r['candidates'] if x['kind']=='continue');joint=next(x for x in r['candidates'] if x['kind']=='joint')
 rows.append({'seed':r['seed'],'index':r['index'],'hypothesis':r['hypothesis']['choices']['relationship'],'layer':r['hypothesis']['choices']['layer'],'selected':r['selected_kind'],'quality':r['selection']['quality'],'changed_fields':[k for k,v in joint['parameters'].items() if v!=ref['parameters'][k]],'joint_RGB_MAE_2_steps':joint['early']['RGB_MAE'],'joint_RGB_MAE_6_steps':joint['late']['RGB_MAE'],'repeated_config':bool(r['repeat_ids']),'ledger_before':r['memory_before_trial_count']})
R_RESULTS=pd.DataFrame(rows);R_RESULTS.to_csv(R_OUT/'pilot_results.csv',index=False)
print('ACTUAL HANDOFF/MEMORY/PREFIX CHECKS:',checks)
display(R_RESULTS)
R_FINAL_COMPARE=[]
for seed in R_SEEDS:
 new=torch.load(R_OUT/f's{seed}_final_latent.pt',map_location='cpu',weights_only=False)
 old=torch.load(R_OLD_OUT/f's{seed}_r09_committed_step99.pt',map_location='cpu',weights_only=False)['latent_after']
 R_FINAL_COMPARE.append({'seed':seed,'max_abs_latent_change_vs_previous_ordinary':float((new-old).abs().max()),'same_latent_as_ordinary':bool(torch.equal(new,old))})
print('FINAL TRAJECTORY CHECK',R_FINAL_COMPARE)
print('STATUS',json.loads((R_OUT/'status.json').read_text()))
r_dump('final_trajectory_check.json',R_FINAL_COMPARE)
r_render()


In [ ]:
# R11. Final audit, including negative results and all endpoint images.
R_PAIR_AUDIT=[]
for x in R_LEDGER:
 obs=x['observed_consequence']
 for horizon in ['early','late']:
  for scale in ['whole','context']:
   if scale in obs.get(horizon,{}):
    q=obs[horizon][scale];p=q.get('parsed') or {}
    R_PAIR_AUDIT.append({'trial':x['trial_id'],'horizon':horizon,'scale':scale,'difference':p.get('difference'),'preference':p.get('preference'),'valid_schema':q.get('valid_schema',False),'pixel_MAE':q.get('pixel_MAE')})
print('Actual paired report differences:',dict(Counter(x['difference'] for x in R_PAIR_AUDIT)))
print('Actual paired report preferences:',dict(Counter(x['preference'] for x in R_PAIR_AUDIT)))
print('Valid paired schemas:',sum(x['valid_schema'] for x in R_PAIR_AUDIT),'/',len(R_PAIR_AUDIT))
print('Actual joint proposals:')
for r in R_ROUNDS:print(r['seed'],r['index'],r['proposed_configuration'])
_first=R_ROUNDS[0];_repeat_hits=r_repeats(_first['seed'],_first['proposed_configuration'])
assert _repeat_hits;R_TESTS['repeat_configuration_detection']=True
r_dump('regression_tests.json',R_TESTS);r_dump('paired_review_audit.json',R_PAIR_AUDIT)
R_SUMMARY=f'''# Repaired controller: completed engineering pilot

The new notebook implements and tests the announced repairs. It does **not** establish an image-quality improvement.

**Implemented and verified:** A completed Jev hypothesis and layer choice are supplied verbatim to the subsequent control-selection request. All prior current-seed branches, including rejected ones, appear in the next request, alongside summaries of the previous experiment. Paired whole-image and enlarged-context views use identical coordinates. Candidate trajectories are observed at two and six steps; exactly the selected five-step prefix is committed. Repeated configurations are detected. Every seed still completes all fifty continuation steps.

**Reviewer test failed:** The initial four-panel layout confused Qwen. Switching to genuinely separate image inputs did not fix its reliability: one identical pair received an A preference, a visibly changed pair was labelled identical, and preference changed under reversed order. Valid output structure was also inconsistent. We verified that distinct processed image tensors reached Qwen (mean absolute patch-tensor difference {R_TESTS['reviewer_input_patch_mean_difference']:.6f}). All raw failed checks are archived; exact pixel identity is computed separately and never counted as a model success.

**Bounded test:** Because these checks failed, the protocol was narrowed before launching diffusion to two preselected seeds and four decisions, with full 50-step completion of both trajectories. It made {R_COUNTS['jev_calls']} Jev calls. All {len(R_LEDGER)} candidate records were retained. The pilot took {json.loads((R_OUT/'status.json').read_text())['pilot_wall_seconds']:.1f} seconds. Visual-call counts include calibration work.

**Result:** All four proposed joint edits changed head gates, identity-perturbation strengths and FreeU. Their two-step RGB effects ranged from {R_RESULTS['joint_RGB_MAE_2_steps'].min():.6f} to {R_RESULTS['joint_RGB_MAE_2_steps'].max():.6f}; six-step effects ranged from {R_RESULTS['joint_RGB_MAE_6_steps'].min():.6f} to {R_RESULTS['joint_RGB_MAE_6_steps'].max():.6f}. These are pixel changes, not anatomical scores. Jev still chose continuation 4/4 times with quality unresolved. Both final latents exactly match the corresponding previous ordinary trajectories. No edited branch was committed and no best-image selection was made.

**What passed:** The tests examined the actual recorded requests and latent states, not just helper-function outputs. They verified hypothesis-call ordering and exact forwarding, fixed layer identity before head choices, inclusion of all earlier same-seed rejected trials, the six-step probe horizon, exact prefix commitment, both complete trajectories, and repeated-configuration detection.

**What remains open:** The corrected information flow alone did not produce useful committed edits. The reviewer still fails simple discriminations, so this pilot cannot tell whether richer visual evidence would let Jev make useful anatomical decisions. Numeric evidence and a changed parameter vector cannot substitute for that result. The next quality experiment needs a reviewer that passes identity and order checks and identifies concrete image relationships, while retaining the same original source. This pilot changes several engineering components together and is not an isolated ablation or evidence that longer loops solve the problem.
'''
(R_OUT/'findings.md').write_text(R_SUMMARY);display(Markdown(R_SUMMARY))
page='<!doctype html><html><head><meta charset="utf-8"><title>Jev repair pilot results</title><style>body{font:16px system-ui;background:#171a20;color:white;margin:20px}.grid{display:grid;grid-template-columns:repeat(3,1fr);gap:16px}img{width:100%}figure{margin:0}pre{white-space:pre-wrap}a{color:#adf}</style></head><body><h1>Repaired controller: all results</h1><p>Both final latents match ordinary continuation exactly. No Jev quality improvement demonstrated.</p><div class="grid">'
for label,im in [('Original seed-123 source',D_SOURCE)]+[(f'Noise seed {s}',Image.open(R_OUT/R_FINALS[s])) for s in R_SEEDS]:
 page+=f'<figure><img src="{w_thumb(im,512)}"><figcaption>{label}</figcaption></figure>'
page+='</div><h2>Findings and tests</h2><pre>'+html.escape(R_SUMMARY)+'</pre><p><a href="live.html">Every probe and chronological progression</a></p></body></html>'
(R_OUT/'results.html').write_text(page)
display(HTML(f'<a target="_blank" href="/files/workspace/crazy_exp/{R_OUT}/results.html">Open the completed pilot results and all endpoint images</a>'))
# Archive final cell source too, so the post-run tests and findings are reproducible.
(R_OUT/'executed_cells.py').write_text('\n\n'.join(s for s in get_ipython().history_manager.input_hist_raw if re.match(r'^# R\d+',s)))
print('COMPLETE. Artifacts:',R_OUT)
